# 🩺 MedOrqis-LLM — Cloud Fine-Tuning Pipeline
### Fine-Tuning  via QLoRA (4-bit NF4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seotarek/medorqis-llm/blob/main/notebooks/MedOrqis_LLM_Training_Colab.ipynb)

This notebook fine-tunes the **MedOrqis-LLM** clinical intelligence model using **QLoRA (4-bit NF4)** on a free **NVIDIA T4 GPU** in Google Colab.

- **Base Foundation Model**: 
- **Clinical Dataset**: [](https://huggingface.co/datasets/seotarek/medorqis-clinical-instructions)
- **Target LoRA Output**: [](https://huggingface.co/seotarek/medorqis-llm)

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate huggingface_hub

## 3. Hugging Face Authentication & Configuration

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Enter your Hugging Face Write Token: ")

login(token=HF_TOKEN)

BASE_MODEL_ID = "Applied-Innovation-Center/Karnak-6B-v1.0"
DATASET_ID = "seotarek/medorqis-clinical-instructions"
OUTPUT_LORA_REPO = "seotarek/medorqis-llm"

print(f"Base Model: {BASE_MODEL_ID}")
print(f"Target Output Hub: {OUTPUT_LORA_REPO}")
print(f"GPU Device: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")

## 4. Load Base Model in 4-bit NF4 Precision

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Load & Format Clinical Dataset

In [ ]:
dataset = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)

def format_example(ex):
    messages = ex.get("messages", [])
    tools = ex.get("tools", None)
    try:
        text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

formatted_dataset = dataset.map(format_example)
print(f"Successfully formatted {len(formatted_dataset)} clinical instruction examples.")

## 6. Supervised Fine-Tuning (SFTTrainer)

In [ ]:
training_args = TrainingArguments(
    output_dir="./medorqis_llm_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=2,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=1536,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config
)

print("Launching QLoRA training on GPU...")
trainer.train()

## 7. Save & Push Trained LoRA Adapters to Hugging Face Hub

In [ ]:
save_dir = "./medorqis-llm-lora-final"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Pushing LoRA adapter weights to: https://huggingface.co/{OUTPUT_LORA_REPO} ...")
trainer.model.push_to_hub(OUTPUT_LORA_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_LORA_REPO, token=HF_TOKEN)
print("
SUCCESS: MedOrqis-LLM adapter weights are live on Hugging Face Hub!")